## 1. Setup and Imports

In [ ]:
import sys
import os
sys.path.insert(0, '/app/src')
sys.path.insert(0, '/app')

from classification import (
    get_classifier,
    load_celebrities_from_json,
    FaceDetector,
    InsightFaceEmbedder,
    InsightFaceClassifier
)
import json

print("✓ Imports successful")

## 2. Load Celebrity Data

In [ ]:
# Load celebrity reference data
celebrity_data = load_celebrities_from_json('/app/data/celebrities.json')
print(f"✓ Loaded {len(celebrity_data)} celebrities")
for celeb in celebrity_data[:3]:
    print(f"  - {celeb['name']}: {celeb['reference_image_path']}")

## 3. Demonstrate Detection-Only (FaceDetector)

Show that FaceDetector works independently with different models

In [ ]:
import cv2

# Create detectors with different models
detector_buffalo_l = FaceDetector(model="buffalo_l")
detector_buffalo_s = FaceDetector(model="buffalo_s")
detector_hog = FaceDetector(model="hog")

print("✓ Created 3 detectors:")
print(f"  1. FaceDetector(buffalo_l) - SCRFD-10GF, highest accuracy")
print(f"  2. FaceDetector(buffalo_s) - SCRFD-500MF, faster, lighter")
print(f"  3. FaceDetector(hog) - Legacy, CPU-friendly")

## 4. Demonstrate Embedding-Only (InsightFaceEmbedder)

Show that InsightFaceEmbedder works independently

In [ ]:
# Create embedders with different models
embedder_buffalo_l = InsightFaceEmbedder(model_name="buffalo_l")
embedder_buffalo_s = InsightFaceEmbedder(model_name="buffalo_s")

print("✓ Created 2 embedders:")
print(f"  1. InsightFaceEmbedder(buffalo_l) - ResNet50, 512-dim")
print(f"  2. InsightFaceEmbedder(buffalo_s) - MobileNet, 512-dim")

## 5. Test Flexible Model Combinations

Now create classifiers with DIFFERENT detection and embedding models

In [ ]:
# Combination 1: Fast detection (buffalo_s) + Accurate embeddings (buffalo_l)
print("Creating classifier combination 1: buffalo_s detection + buffalo_l embeddings...")
try:
    classifier_fast_accurate = InsightFaceClassifier(
        name="fast_accurate",
        celebrity_data=celebrity_data,
        detection_model="buffalo_s",  # Lightweight detection
        embedding_model="buffalo_l",  # Accurate embeddings
        threshold=0.6
    )
    print(f"✓ {classifier_fast_accurate.name}")
    print(f"  Detection: {classifier_fast_accurate.detection_model}")
    print(f"  Embeddings: {classifier_fast_accurate.embedding_model}")
except Exception as e:
    print(f"✗ Error: {e}")

In [ ]:
# Combination 2: Accurate detection (buffalo_l) + Lightweight embeddings (buffalo_s)
print("Creating classifier combination 2: buffalo_l detection + buffalo_s embeddings...")
try:
    classifier_accurate_fast = InsightFaceClassifier(
        name="accurate_fast",
        celebrity_data=celebrity_data,
        detection_model="buffalo_l",  # Accurate detection
        embedding_model="buffalo_s",  # Lightweight embeddings
        threshold=0.6
    )
    print(f"✓ {classifier_accurate_fast.name}")
    print(f"  Detection: {classifier_accurate_fast.detection_model}")
    print(f"  Embeddings: {classifier_accurate_fast.embedding_model}")
except Exception as e:
    print(f"✗ Error: {e}")

In [ ]:
# Combination 3: HOG detection + InsightFace embeddings
print("Creating classifier combination 3: hog detection + buffalo_l embeddings...")
try:
    classifier_hog_insightface = InsightFaceClassifier(
        name="hog_insightface",
        celebrity_data=celebrity_data,
        detection_model="hog",        # Legacy HOG detection
        embedding_model="buffalo_l",  # Modern InsightFace embeddings
        threshold=0.6
    )
    print(f"✓ {classifier_hog_insightface.name}")
    print(f"  Detection: {classifier_hog_insightface.detection_model}")
    print(f"  Embeddings: {classifier_hog_insightface.embedding_model}")
except Exception as e:
    print(f"✗ Error: {e}")

## 6. Factory Function Support for Flexible Combinations

The factory function now supports specifying combinations

In [ ]:
# Using factory with different specs
print("Testing factory function support:\n")

# Default: buffalo_l + buffalo_l
print("1. get_classifier('insightface', celebrity_data)")
try:
    c1 = get_classifier('insightface', celebrity_data)
    print(f"   ✓ Detection: {c1.detection_model}, Embeddings: {c1.embedding_model}\n")
except Exception as e:
    print(f"   ✗ {e}\n")

# Specific model
print("2. get_classifier('insightface_buffalo_s', celebrity_data)")
try:
    c2 = get_classifier('insightface_buffalo_s', celebrity_data)
    print(f"   ✓ Detection: {c2.detection_model}, Embeddings: {c2.embedding_model}\n")
except Exception as e:
    print(f"   ✗ {e}\n")

# Custom combination
print("3. get_classifier('insightface_buffalo_s+buffalo_l', celebrity_data)")
try:
    c3 = get_classifier('insightface_buffalo_s+buffalo_l', celebrity_data)
    print(f"   ✓ Detection: {c3.detection_model}, Embeddings: {c3.embedding_model}\n")
except Exception as e:
    print(f"   ✗ {e}\n")

## 7. Architecture Benefits

This refactoring enables several important capabilities:

In [ ]:
benefits = """
MODULAR ARCHITECTURE BENEFITS:

1. FLEXIBLE MODEL COMBINATIONS
   - Mix any detection model with any embedding model
   - buffalo_s detection + buffalo_l embeddings (speed + accuracy)
   - cnn detection + buffalo_l embeddings (legacy + modern)
   - hog detection + insightface embeddings

2. INDEPENDENT OPTIMIZATION
   - Fine-tune detection and embedding separately
   - Use smaller models for detection if speed matters
   - Use larger models for embeddings if accuracy matters

3. EASY TESTING
   - Compare different detection models with same embeddings
   - Compare different embedding models with same detection
   - Systematic evaluation of model combinations

4. MAINTAINABILITY
   - FaceDetector responsible only for detection
   - InsightFaceEmbedder responsible only for embeddings
   - InsightFaceClassifier orchestrates the two-phase pipeline
   - Cleaner code, easier to debug, simpler to extend

5. REUSABILITY
   - FaceDetector can be used standalone for object detection
   - InsightFaceEmbedder can be used standalone for embedding extraction
   - Other classifiers can use InsightFaceEmbedder with different detectors

6. GPU/CPU MANAGEMENT
   - Load detection and embedding models on same or different devices
   - Reduce memory footprint by using lighter models
   - Better resource utilization
"""
print(benefits)

## 8. Summary

The refactored architecture successfully separates concerns:
- **FaceDetector**: Face detection (supports buffalo_l, buffalo_m, buffalo_s, antelopev2, cnn, hog)
- **InsightFaceEmbedder**: Embedding/identification (supports buffalo_l, buffalo_m, buffalo_s, antelopev2)
- **InsightFaceClassifier**: Orchestrates the two-phase pipeline

This enables flexible testing combinations without changing code.